In [1]:
from typing import List, Optional
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os, warnings, logging, json, sys, yaml, random, re
from pathlib import Path
from collections import deque

os.environ["TRANSFORMERS_VERBOSITY"] = "error"
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers.generation").setLevel(logging.ERROR)
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

SAVE_DIR = Path("./trajectories")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
success_steps_list = []
success_trajectories = []
failure_trajectories = []

def save_all_json():
    (SAVE_DIR / "success_steps.json").write_text(
        json.dumps(success_steps_list, indent=2, ensure_ascii=False)
    )
    (SAVE_DIR / "success.json").write_text(
        json.dumps(success_trajectories, indent=2, ensure_ascii=False)
    )
    (SAVE_DIR / "failure.json").write_text(
        json.dumps(failure_trajectories, indent=2, ensure_ascii=False)
    )

# --- Hugging Face LLM ---
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto",
)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

def llm(prompt: str, stop: List[str] = ["\n"], max_new_tokens: int = 100) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.0,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    completion = text[len(prompt):]
    if stop:
        for s in stop:
            if s in completion:
                completion = completion.split(s)[0]
    return completion.strip()

import alfworld
from alfworld.agents.environment import get_environment

with open("base_config.yaml") as f:
    config = yaml.safe_load(f)

split = "eval_out_of_distribution"
EnvClass = get_environment(config["env"]["type"])
env = EnvClass(config, train_eval=split)
env = env.init_env(batch_size=1)

def process_ob(ob: str) -> str:
    if ob.startswith('You arrive at loc '):
        ob = ob[ob.find('. ')+2:]
    return ob

def unpack_info(info):
    return info[0] if isinstance(info, list) else info

def to_string_actions(x) -> List[str]:
    out = []
    def rec(obj):
        if obj is None: return
        if isinstance(obj, (list, tuple, set)):
            for z in obj: rec(z)
        elif isinstance(obj, str):
            out.append(obj)
        else:
            try: out.append(str(obj))
            except: pass
    rec(x)
    seen, flat = set(), []
    for s in out:
        if s not in seen:
            seen.add(s); flat.append(s)
    return flat

def get_valid_actions(info) -> List[str]:
    info = unpack_info(info)
    raw = info.get("admissible_commands", [])
    return to_string_actions(raw)

REFLEXION_CFG = {
    "enable": True,
    "checkpoints": [60],   
    "stall_win": 5,
    "stall_nothing_happens": 2,
    "mem_max": 3,
}

UNCERTAINTY_CFG = {
    "enable": True,
    "entropy_thr": 1.5,
    "min_actions": 4,
}

@torch.no_grad()
def seq_logprob(prefix: str, candidate: str) -> float:
    full_txt = prefix + candidate
    enc_full = tokenizer(full_txt, return_tensors="pt")
    enc_pref = tokenizer(prefix, return_tensors="pt")
    enc_full = {k: v.to(model.device) for k, v in enc_full.items()}
    L_full = enc_full["input_ids"].shape[1]
    L_pref = enc_pref["input_ids"].shape[1]
    out = model(**enc_full)
    logits = out.logits.float()
    logp = 0.0
    for t in range(L_pref, L_full):
        prev_logits = logits[0, t-1, :]
        log_probs = torch.log_softmax(prev_logits, dim=-1)
        token_id = enc_full["input_ids"][0, t]
        logp += log_probs[token_id].item()
    return float(logp)

def action_distribution(prefix: str, admissible: List[str]):
    if len(admissible) < UNCERTAINTY_CFG["min_actions"]:
        return [], 0.0, 0.0, 0.0
    ll = []
    for a in admissible:
        try: ll.append(seq_logprob(prefix, a))
        except Exception: ll.append(-1e9)
    ll_t = torch.tensor(ll)
    prob = torch.softmax(ll_t, dim=0)
    H = -(prob * torch.log(prob + 1e-8)).sum().item()
    if prob.numel() > 1:
        top2 = torch.topk(prob, 2).values
        margin = (top2[0] - top2[1]).item()
    else:
        margin = prob.max().item()
    return prob.tolist(), float(H), float(margin), float(prob.max().item())

def should_call_advisor_by_uncertainty(prefix: str, admissible: List[str], to_print=False) -> bool:
    if not UNCERTAINTY_CFG["enable"] or len(admissible) < UNCERTAINTY_CFG["min_actions"]:
        return False
    _, H, margin, pmax = action_distribution(prefix, admissible)
    trigger = (H > UNCERTAINTY_CFG["entropy_thr"])
    if to_print and trigger:
        print(f"[Uncertainty] H={H:.3f}, margin={margin:.3f}, pmax={pmax:.3f} → Advisor!")
    return trigger

class ReflexionMemory:
    def __init__(self, maxlen=3):
        self.slots = deque(maxlen=maxlen)
    def add(self, lesson: str):
        if lesson and lesson.strip():
            self.slots.append(lesson.strip())
    def prompt_block(self) -> str:
        if not self.slots: return ""
        lessons = "\n".join(f"- {m}" for m in list(self.slots)[-self.slots.maxlen:])
        return f"## Prior self-reflections (use these to avoid past mistakes)\n{lessons}\n"

REFLEXION_MEM = ReflexionMemory(maxlen=REFLEXION_CFG["mem_max"])

def _extract_json_block(txt: str) -> dict:
    try:
        s = txt.find("{"); e = txt.rfind("}")
        if s != -1 and e != -1 and e > s:
            return json.loads(txt[s:e+1])
    except Exception:
        pass
    return {}

SYS2_STEP_PROMPT = """You are System-2, a slow but precise analyst for an ALFWorld ReAct agent.
You will receive: (1) TASK, (2) PRIOR REFLECTIONS, (3) TRAJECTORY SO FAR (Act/Obs pairs),
and (4) CURRENT ADMISSIBLE ACTIONS.
Focus on the facts. Diagnose mistakes, extract key facts, propose a micro-plan,
and output ONE best NEXT ACTION strictly chosen from ADMISSIBLE.
If everything is fine, say advice "you are doing well. keep doing".
Return JSON only:
{
  "rationale": "...",
  "facts": ["...", "..."],
  "plan": ["step1", "step2", "step3"],
  "next_action": "exact string from ADMISSIBLE",
  "advice": "one-sentence coaching"
}
"""

def system2_step_advice(task_line: str, long_mem_block: str,
                        traj_snippet: str, admissible: List[str]) -> dict:
    admissible_block = "\n".join(f"- {a}" for a in admissible)
    prompt = (
        f"{SYS2_STEP_PROMPT}\n\n"
        f"TASK:\n{task_line}\n\n"
        f"PRIOR REFLECTIONS:\n{long_mem_block or '(none)'}\n\n"
        f"TRAJECTORY SO FAR (latest first up to ~25 lines):\n{traj_snippet}\n\n"
        f"CURRENT ADMISSIBLE ACTIONS:\n{admissible_block}\n\n"
        f"JSON:"
    )
    text = llm(prompt, stop=[], max_new_tokens=400)
    data = _extract_json_block(text)
    if data.get("next_action") not in admissible:
        data["next_action"] = None
    return data

SYS2_TRIAL_PROMPT = """You are System-2. Write ONE short lesson (<=25 words) that would have
prevented the failure or made the success faster, given TASK and the FULL TRAJECTORY.
Return JSON: {"lesson": "..."}"""

def system2_trial_reflection(task_line: str, full_traj_text: str, success: bool) -> str:
    tag = "SUCCESS" if success else "FAILURE"
    prompt = f"{SYS2_TRIAL_PROMPT}\n\nTAG:{tag}\nTASK:\n{task_line}\n\nTRAJECTORY:\n{full_traj_text}\n\nJSON:"
    text = llm(prompt, stop=[], max_new_tokens=400)
    data = _extract_json_block(text)
    return (data.get("lesson") or "").strip()

def _make_traj_snippet(steps: List[dict], limit_chars=1800) -> str:
    parts = []
    for st in reversed(steps[-40:]):
        if st["type"] == "think":
            parts.append(f"think: {st['llm']}")
        else:
            parts.append(f"Act: {st['exec']}\nObs: {st['obs']}")
    s = "\n".join(parts[::-1])
    return s[-limit_chars:]

def _should_call_sys2(i: int, recent_obs: List[str]) -> bool:
    if i in REFLEXION_CFG["checkpoints"]:
        return True
    low = [o.lower() for o in recent_obs[-REFLEXION_CFG["stall_win"]:]]
    if sum("nothing happens" in x for x in low) >= REFLEXION_CFG["stall_nothing_happens"]:
        return True
    return False

def reask_pick_from_admissible(prefix_text: str, admissible: List[str]) -> Optional[str]:
    if not admissible: return None
    choices = "\n".join(f"- {a}" for a in admissible)
    prompt = (
        f"{prefix_text}\n\n"
        f"Choose your next action strictly from the following list. "
        f"Respond with the exact string of ONE action only, no extra words.\n{choices}\n\n>"
    )
    ans = llm(prompt, stop=["\n"], max_new_tokens=40).strip()
    return ans if ans in admissible else None

folder = './prompts/'
prompt_file = 'alfworld_3prompts_1.json'
with open(os.path.join(folder, prompt_file), 'r') as f:
    d = json.load(f)

def alfworld_run(prompt: str, to_print: bool = True, ob: str = '', info=None,
                 episode_name: str = "", split_name: str = "") -> tuple:
    
    task_line = ""
    for line in ob.splitlines():
        if line.lower().startswith("your task is to:"):
            task_line = line
            break

    init_prompt = REFLEXION_MEM.prompt_block() + prompt + ob + '\n>'
    prompt_ctx = ''
    if to_print:
        print("[S1] Initial Obs:\n" + ob); sys.stdout.flush()

    traj = {
        "episode_name": episode_name,
        "split": split_name,
        "success": None,
        "total_steps": 0,
        "steps": []
    }
    env_steps = 0
    curr_info = info
    recent_obs_buf: List[str] = []

    for i in range(1, 50):
        # System-1 제안
        llm_input = init_prompt + prompt_ctx
        action_txt = llm(llm_input, stop=['\n'], max_new_tokens=100).strip()

        if action_txt.lower().startswith('think:'):
            if to_print:
                print(f'[S1] Act {i}: {action_txt}\n[S1] Obs {i}: OK.')
            prompt_ctx += f' {action_txt}\nOK.\n>'
            traj["steps"].append({
                "i": i, "type": "think", "llm": action_txt, "exec": None,
                "obs": "OK.", "reward": None, "done": False
            })
            continue

        vacts_now = get_valid_actions(curr_info) if curr_info is not None else []
        call_sys2 = (
            REFLEXION_CFG["enable"] and (
                _should_call_sys2(i, recent_obs_buf) or
                should_call_advisor_by_uncertainty(llm_input, vacts_now, to_print=to_print)
            )
        )
        if call_sys2 and vacts_now:
            traj_snip = _make_traj_snippet(traj["steps"])
            prior_block = REFLEXION_MEM.prompt_block()
            advice = system2_step_advice(task_line, prior_block, traj_snip, vacts_now)
            if to_print:
                print(f"[S2] Trigger at step {i}.")
                print(f"[S2] Rationale: {advice.get('rationale','')}")
                print(f"[S2] Plan: {advice.get('plan',[])}")
                print(f"[S2] Advice: {advice.get('advice','')}")
                print(f"[S2] Next action (if any): {advice.get('next_action')}")
            inject = " ".join(filter(None, [
                advice.get("rationale",""),
                ("Plan: " + "; ".join(advice.get("plan", []))) if advice.get("plan") else "",
                ("Tip: " + advice.get("advice","")) if advice.get("advice") else ""
            ])).strip()
            if inject:
                prompt_ctx += f" ADVICE: {inject}\nACK.\n>"
            if advice.get("next_action") in vacts_now:
                action_txt = advice["next_action"]

        if curr_info is None:
            obs, reward, done, curr_info = env.step(["look around"])
            env_steps += 1
            ob0 = process_ob(obs[0])
            traj["steps"].append({
                "i": i, "type": "act", "llm": "look around (auto-init)", "exec": "look around",
                "obs": ob0, "reward": reward[0] if isinstance(reward, list) else reward,
                "done": done[0] if isinstance(done, list) else done
            })
            if to_print:
                print(f"[S1] Auto-init: look around\n[S1] Obs: {ob0}")
            recent_obs_buf.append(ob0); recent_obs_buf[:] = recent_obs_buf[-12:]

        vacts = get_valid_actions(curr_info)
        act = action_txt

        if vacts and act not in vacts:
            picked = reask_pick_from_admissible(init_prompt + prompt_ctx, vacts)
            if picked is not None:
                act = picked
            else:
                act = random.choice(vacts)

        obs, reward, done, curr_info = env.step([act])
        env_steps += 1
        ob_str = process_ob(obs[0])
        if to_print:
            print(f'[S1] Act {i}: {act}\n[S1] Obs {i}: {ob_str}')
        prompt_ctx += f' {act}\n{ob_str}\n>'
        recent_obs_buf.append(ob_str); recent_obs_buf[:] = recent_obs_buf[-12:]

        traj["steps"].append({
            "i": i, "type": "act", "llm": action_txt, "exec": act,
            "obs": ob_str,
            "reward": reward[0] if isinstance(reward, list) else reward,
            "done": done[0] if isinstance(done, list) else done
        })

        if done[0]:
            won = unpack_info(curr_info)['won'][0]
            # trial-level reflection
            full_traj_text = "\n".join(
                f"{'think' if s['type']=='think' else 'act'}: {s.get('llm') or s.get('exec')}\nobs: {s['obs']}"
                for s in traj["steps"]
            )
            lesson = system2_trial_reflection(task_line, full_traj_text, success=bool(won))
            if lesson:
                REFLEXION_MEM.add(lesson)
                if to_print: print(f"[S2] Trial reflection: {lesson}")
            traj["success"] = bool(won); traj["total_steps"] = env_steps
            return won, traj, env_steps

    won = 0
    full_traj_text = "\n".join(
        f"{'think' if s['type']=='think' else 'act'}: {s.get('llm') or s.get('exec')}\nobs: {s['obs']}"
        for s in traj["steps"]
    )
    lesson = system2_trial_reflection(task_line, full_traj_text, success=False)
    if lesson:
        REFLEXION_MEM.add(lesson)
        if to_print: print(f"[S2] Trial reflection: {lesson}")
    traj["success"] = False
    traj["total_steps"] = env_steps
    return won, traj, env_steps

prefixes = {
    'pick_and_place': 'put',
    'pick_clean_then_place': 'clean',
    'pick_heat_then_place': 'heat',
    'pick_cool_then_place': 'cool',
    'look_at_obj': 'examine',
    'pick_two_obj': 'puttwo'
}
cnts = [0] * 6
rs = [0] * 6

for _ in range(134):
    ob, info = env.reset()
    ob = '\n'.join(ob[0].split('\n\n')[1:])
    name = '/'.join(info['extra.gamefile'][0].split('/')[-3:-1])
    print(name)
    for i, (k, v) in enumerate(prefixes.items()):
        if name.startswith(k):
            prompt = 'Interact with a household to solve a task. Here are two examples.\n' + \
                     d[f'react_{v}_1'] + d[f'react_{v}_0'] + '\nHere is the task.\n'
            print(k, v)
            r, traj, steps_used = alfworld_run(
                prompt, ob=ob, info=info,
                episode_name=name, split_name=split
            )
            rs[i] += r
            cnts[i] += 1
            if r == 1:
                success_steps_list.append(steps_used)
                success_trajectories.append(traj)
            else:
                failure_trajectories.append(traj)
            break
    total_cnt = max(1, sum(cnts))
    print(_+1, 'r', r, 'rs', rs, 'cnts', cnts, 'sum(rs)/sum(cnts)', sum(rs) / total_cnt)
    print('------------\n')

save_all_json()


ModuleNotFoundError: No module named 'transformers'